In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

def generate_synthetic_cable(bg_image, num_points=100):
    """
    Generates a synthetic cable over a background image and returns
    the augmented RGB image and the exact 1-pixel binary mask.
    """
    h, w = bg_image.shape[:2]

    # 1. Define Bezier Control Points (Start, Sag/Control, End)
    # Randomizing these in a loop will give you infinite variations of sag
    p0 = np.array([0, np.random.randint(h//4, h//2)])          # Left edge
    p2 = np.array([w, np.random.randint(h//4, h//2)])          # Right edge

    # The control point defines the droop/sag
    sag_offset = np.random.randint(50, 200)
    p1 = np.array([w // 2, max(p0[1], p2[1]) + sag_offset])    # Middle droop

    # 2. Calculate the Bezier Curve coordinates
    t = np.linspace(0, 1, num_points)
    # B(t) = (1-t)^2*P0 + 2(1-t)t*P1 + t^2*P2
    curve_x = ((1-t)**2 * p0[0] + 2*(1-t)*t * p1[0] + t**2 * p2[0]).astype(np.int32)
    curve_y = ((1-t)**2 * p0[1] + 2*(1-t)*t * p1[1] + t**2 * p2[1]).astype(np.int32)

    pts = np.vstack((curve_x, curve_y)).T
    pts = pts.reshape((-1, 1, 2))

    # 3. Create the Flawless Binary Mask
    mask = np.zeros((h, w), dtype=np.uint8)
    # Thickness is STRICTLY 1 pixel (or 2 if simulating distance)
    cv2.polylines(mask, [pts], isClosed=False, color=255, thickness=1)

    # 4. Render on the RGB Background
    synthetic_rgb = bg_image.copy()

    # We draw it slightly thicker on a temp layer to apply blur for realism
    temp_layer = np.zeros_like(synthetic_rgb)
    cable_color = (30, 30, 30) # Dark grey/black cable
    cv2.polylines(temp_layer, [pts], isClosed=False, color=cable_color, thickness=2)

    # Apply Gaussian Blur to the line to simulate camera focus/depth
    temp_layer = cv2.GaussianBlur(temp_layer, (3, 3), 0)

    # Blend the blurred cable into the background
    cable_pixels = temp_layer > 0
    synthetic_rgb[cable_pixels] = temp_layer[cable_pixels]

    # Optional: Add slight Gaussian noise to the whole image to unify the synthetic line
    noise = np.random.normal(0, 5, synthetic_rgb.shape).astype(np.uint8)
    synthetic_rgb = cv2.add(synthetic_rgb, noise)

    # Normalize mask to strictly 0.0 and 1.0 per your pipeline specs
    mask_normalized = mask.astype(np.float32) / 255.0

    return synthetic_rgb, mask_normalized

# --- Sanity Check Execution ---
# Create a dummy noisy background (Replace this with a crop from your actual images)
dummy_bg = np.random.randint(150, 255, (512, 512, 3), dtype=np.uint8)

syn_image, syn_mask = generate_synthetic_cable(dummy_bg)

# Visual check using your existing matplotlib setup logic
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(syn_image)
ax[0].set_title("Synthetic RGB (Input)")
ax[1].imshow(syn_mask, cmap='gray')
ax[1].set_title("Flawless Ground Truth Mask")
plt.show()

In [ ]:
import cv2
import numpy as np
import random
import matplotlib.pyplot as plt

def extract_pure_backgrounds(image_path, mask_path, output_dir, crop_size=512, crops_per_image=5):
    """
    Extracts patches of an image strictly where the mask is completely black.
    """
    img = cv2.imread(image_path)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    # --- The Fail-Fast Safeguard ---
    if img is None:
        raise FileNotFoundError(f"OpenCV could not find the image at: {image_path}")
    if mask is None:
        raise FileNotFoundError(f"OpenCV could not find the mask at: {mask_path}")

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    valid_crops = []
    attempts = 0
    max_attempts = 100

    while len(valid_crops) < crops_per_image and attempts < max_attempts:
        y = random.randint(0, h - crop_size)
        x = random.randint(0, w - crop_size)

        mask_patch = mask[y:y+crop_size, x:x+crop_size]

        if np.sum(mask_patch) == 0:
            img_patch = img[y:y+crop_size, x:x+crop_size]
            valid_crops.append(img_patch)

            save_path = os.path.join(output_dir, f"{os.path.basename(image_path)}_bg_{len(valid_crops)}.jpg")
            cv2.imwrite(save_path, cv2.cvtColor(img_patch, cv2.COLOR_RGB2BGR))

        attempts += 1

    return valid_crops

# --- Blank Colab Test Execution ---
# Grab the first image/mask pair to test the logic visually
# Grab the first image to test
sample_img_name = os.listdir(IMAGES_DIR)[0]
sample_img_path = os.path.join(IMAGES_DIR, sample_img_name)

# Fix the naming convention map: "image_0.jpg" -> "mask_image_0.png"
mask_name = f"mask_{sample_img_name.replace('.jpg', '.png')}"
sample_mask_path = os.path.join(MASKS_DIR, mask_name)

print(f"Testing Image: {sample_img_path}")
print(f"Testing Mask: {sample_mask_path}")

extracted_bgs = extract_pure_backgrounds(sample_img_path, sample_mask_path, OUTPUT_DIR, crop_size=256, crops_per_image=2)

# Sanity check visualization
if extracted_bgs:
    fig, axes = plt.subplots(1, len(extracted_bgs), figsize=(10, 5))
    if len(extracted_bgs) == 1:
        axes = [axes]
    for i, bg in enumerate(extracted_bgs):
        axes[i].imshow(bg)
        axes[i].set_title(f"Pure Background Canvas {i+1}")
    plt.show()
else:
    print("Could not find any 256x256 areas without the cable. Try a smaller crop size.")

In [ ]:
import os
import cv2
import numpy as np
import random
from tqdm import tqdm # For a progress bar

# --- Configuration & Paths ---
BASE_DIR = '/content/drive/MyDrive/Robotics_Project'
IMAGES_DIR = os.path.join(BASE_DIR, 'Raw_Data/images')
MASKS_DIR = os.path.join(BASE_DIR, 'Raw_Data/masks')

# The final output destination for the PyTorch DataLoader
FINAL_DATASET_DIR = os.path.join(BASE_DIR, 'Training_Dataset')
FINAL_IMAGES_DIR = os.path.join(FINAL_DATASET_DIR, 'images')
FINAL_MASKS_DIR = os.path.join(FINAL_DATASET_DIR, 'masks')

os.makedirs(FINAL_IMAGES_DIR, exist_ok=True)
os.makedirs(FINAL_MASKS_DIR, exist_ok=True)

# Parameters
CROP_SIZE = 256
CROPS_PER_IMAGE = 10 # 45 images * 10 crops = 450 backgrounds
SYNTHETICS_PER_CROP = 5 # 450 backgrounds * 5 cable variations = 2,250 final training images

# --- 1. Background Extraction Logic ---
def get_clean_backgrounds(image_path, mask_path, size, count):
    img = cv2.imread(image_path)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if img is None or mask is None:
        return []

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    valid_crops = []
    attempts = 0

    while len(valid_crops) < count and attempts < 200:
        y = random.randint(0, h - size)
        x = random.randint(0, w - size)
        if np.sum(mask[y:y+size, x:x+size]) == 0:
            valid_crops.append(img[y:y+size, x:x+size])
        attempts += 1
    return valid_crops

# --- 2. Synthetic Cable Logic ---
def inject_synthetic_cable(bg_image):
    h, w = bg_image.shape[:2]

    # 1. Define the Bezier curve (Sag calculation)
    p0 = np.array([0, random.randint(h//4, int(h*0.75))])
    p2 = np.array([w, random.randint(h//4, int(h*0.75))])
    sag_offset = random.randint(30, 150)
    p1 = np.array([w // 2, max(p0[1], p2[1]) + sag_offset])

    t = np.linspace(0, 1, 100)
    curve_x = ((1-t)**2 * p0[0] + 2*(1-t)*t * p1[0] + t**2 * p2[0]).astype(np.int32)
    curve_y = ((1-t)**2 * p0[1] + 2*(1-t)*t * p1[1] + t**2 * p2[1]).astype(np.int32)

    pts = np.vstack((curve_x, curve_y)).T.reshape((-1, 1, 2))

    # 2. Generate 1-pixel exact ground truth mask
    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.polylines(mask, [pts], isClosed=False, color=255, thickness=1)

    # 3. High-Visibility RGB Rendering
    synthetic_rgb = bg_image.copy()

    # Use a distinct, very dark color to stand out against the static (RGB format)
    cable_color = (0, 0, 0)

    # Draw directly onto the image using Anti-Aliasing (cv2.LINE_AA)
    # This keeps the line crisp and visible without adding artificial blur or noise
    cv2.polylines(synthetic_rgb, [pts], isClosed=False, color=cable_color, thickness=2, lineType=cv2.LINE_AA)

    return synthetic_rgb, mask
# --- 3. Master Execution Loop ---
raw_images = [f for f in os.listdir(IMAGES_DIR) if f.endswith('.jpg')]
total_generated = 0

print(f"Starting pipeline on {len(raw_images)} raw images...")

for img_name in tqdm(raw_images):
    img_path = os.path.join(IMAGES_DIR, img_name)
    mask_name = f"mask_{img_name.replace('.jpg', '.png')}"
    mask_path = os.path.join(MASKS_DIR, mask_name)

    # Get clean backgrounds
    backgrounds = get_clean_backgrounds(img_path, mask_path, CROP_SIZE, CROPS_PER_IMAGE)

    for bg_idx, bg_img in enumerate(backgrounds):
        for syn_idx in range(SYNTHETICS_PER_CROP):
            syn_rgb, syn_mask = inject_synthetic_cable(bg_img)

            # Format filename: origName_bgX_synY
            base_name = img_name.replace('.jpg', '')
            file_identifier = f"{base_name}_bg{bg_idx}_syn{syn_idx}"

            # Save RGB (Convert RGB to BGR for cv2)
            cv2.imwrite(os.path.join(FINAL_IMAGES_DIR, f"{file_identifier}.jpg"), cv2.cvtColor(syn_rgb, cv2.COLOR_RGB2BGR))
            # Save Mask
            cv2.imwrite(os.path.join(FINAL_MASKS_DIR, f"{file_identifier}.png"), syn_mask)

            total_generated += 1

print(f"\nPipeline Complete! Generated {total_generated} flawless synthetic images and masks.")

In [ ]:
# split dataset into train and validation:

import os
import shutil
import random
from pathlib import Path

# Base paths based on your previous synthetic generation
BASE_DIR = '/content/drive/MyDrive/Robotics_Project'
SYNTH_IMG_DIR = os.path.join(BASE_DIR, 'Training_Dataset/images')
SYNTH_MASK_DIR = os.path.join(BASE_DIR, 'Training_Dataset/masks')
OUTPUT_SPLIT_DIR = os.path.join(BASE_DIR, 'Split_Dataset')

def split_dataset(image_dir, mask_dir, output_dir, split_ratio=0.8, seed=42):
    random.seed(seed)

    images_path = Path(image_dir)
    mask_path = Path(mask_dir)
    output_path = Path(output_dir)

    for split in ['train', 'val']:
        (output_path / split / 'images').mkdir(parents=True, exist_ok=True)
        (output_path / split / 'masks').mkdir(parents=True, exist_ok=True)

    image_files = sorted(list(images_path.rglob("*.jpg")))
    mask_files = sorted(list(mask_path.rglob("*.png")))

    if len(image_files) != len(mask_files):
        print("Error: Images and masks mismatch. Check your generation folders.")
        return

    paired_files = list(zip(image_files, mask_files))
    random.shuffle(paired_files)

    train_size = int(len(paired_files) * split_ratio)

    train_pairs = paired_files[:train_size]
    val_pairs = paired_files[train_size:]

    print(f"Total pairs: {len(paired_files)}")
    print(f"Training pairs: {len(train_pairs)}")
    print(f"Validation pairs: {len(val_pairs)}\n")

    def copy_files(pairs, split_name):
        for img_path, mask_path in pairs:
            dest_img = output_path / split_name / 'images' / img_path.name
            dest_mask = output_path / split_name / 'masks' / mask_path.name

            shutil.copy2(img_path, dest_img)
            shutil.copy2(mask_path, dest_mask)

    copy_files(train_pairs, 'train')
    copy_files(val_pairs, 'val')
    print("Dataset successfully split!")

# Execute the split
split_dataset(image_dir=SYNTH_IMG_DIR, mask_dir=SYNTH_MASK_DIR, output_dir=OUTPUT_SPLIT_DIR)

In [ ]:
import os

# --- KAGGLE PATH CONFIGURATION ---

# 1. READ-ONLY PATHS (Where your uploaded dataset lives)
# Note: You will need to check the exact folder name Kaggle gave your dataset 
# by clicking the 'Data' tab on the right side of the notebook.
INPUT_DIR = '/kaggle/input/cable-robot-dataset/Robotics_Project'

RAW_IMG_DIR = os.path.join(INPUT_DIR, 'Raw_Data/images/')
RAW_MASK_DIR = os.path.join(INPUT_DIR, 'Raw_Data/masks/')

# If you already split the dataset before moving to Kaggle:
train_img_dir = os.path.join(INPUT_DIR, 'Split_Dataset/train/images/')
train_mask_dir = os.path.join(INPUT_DIR, 'Split_Dataset/train/masks/')
val_img_dir = os.path.join(INPUT_DIR, 'Split_Dataset/val/images/')
val_mask_dir = os.path.join(INPUT_DIR, 'Split_Dataset/val/masks/')

# 2. WRITABLE PATHS (Where you save things during training)
WORKING_DIR = '/kaggle/working/'
FT_IMG_DIR = os.path.join(WORKING_DIR, 'images/')
FT_MASK_DIR = os.path.join(WORKING_DIR, 'masks/')

In [ ]:
# model.py:

import torch
import torch.nn as nn

class DilatedCableNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super(DilatedCableNet, self).__init__()

        self.frontend = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True)
        )

        self.context = nn.Sequential(
            nn.Conv2d(32, 32, kernel_size=3, dilation=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 32, kernel_size=3, dilation=2, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 32, kernel_size=3, dilation=4, padding=4),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 32, kernel_size=3, dilation=8, padding=8),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 32, kernel_size=3, dilation=16, padding=16),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 32, kernel_size=3, dilation=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True)
        )

        self.classifier = nn.Conv2d(32, out_channels, kernel_size=1)

    def forward(self, x):
        x = self.frontend(x)
        x = self.context(x)
        logits = self.classifier(x)
        return logits

In [ ]:
# customDataset.py:

import torch
from torch.utils.data import Dataset
from PIL import Image
import os
import numpy as np

class CableRobotDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None):
        super(CableRobotDataset, self).__init__()
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.transform = transform

        self.images = [f for f in os.listdir(image_dir) if f.endswith('.jpg')]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        image_file_name = self.images[index]
        image_path = os.path.join(self.image_dir, image_file_name)

        mask_file_name = image_file_name.replace('.jpg', '.png')
        mask_path = os.path.join(self.mask_dir, mask_file_name)

        # REVERTED: Standard 3-Channel RGB loading
        image = np.array(Image.open(image_path).convert("RGB"))
        
        # Load mask and ensure it is a 2D float array
        mask = np.array(Image.open(mask_path).convert("L"), dtype=np.float32)
        mask = mask / 255.0

        if self.transform is not None:
            augmentations = self.transform(image=image, mask=mask)
            image = augmentations["image"]
            mask = augmentations["mask"]

        return image, mask

In [ ]:
# utils.py:

import torch
import torchvision
from torch.utils.data import DataLoader
import os

def save_checkpoint(state, filename='model.pth.tar'):
    print("=> Saving checkpoint")
    torch.save(state, filename)

def load_checkpoint(checkpoint, model, optimizer):
    print("=> Loading checkpoint")
    model.load_state_dict(checkpoint['state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer'])
    return checkpoint.get('epoch', 0)

def get_loaders(train_image_dir, train_mask_dir, train_transform,
                val_image_dir, val_mask_dir, val_transform,
                batch_size, pin_memory=True, num_workers=2):

    train_data = CableRobotDataset(image_dir=train_image_dir, mask_dir=train_mask_dir, transform=train_transform)
    train_loader = DataLoader(dataset=train_data, shuffle=True, batch_size=batch_size, pin_memory=pin_memory, num_workers=num_workers)

    val_data = CableRobotDataset(image_dir=val_image_dir, mask_dir=val_mask_dir, transform=val_transform)
    val_loader = DataLoader(dataset=val_data, batch_size=batch_size, shuffle=False, pin_memory=pin_memory, num_workers=num_workers)

    return train_loader, val_loader

def check_accuracy(loader, model, device='cuda'):
    num_correct = 0
    num_pixels = 0
    dice_score = 0
    model.eval()

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            # Reinstated .unsqueeze(1) to convert [Batch, H, W] -> [Batch, 1, H, W]
            y = y.to(device).unsqueeze(1)

            preds = torch.sigmoid(model(x))
            preds = (preds > 0.5).float()

            num_correct += (preds == y).sum()
            num_pixels += torch.numel(preds)
            dice_score += (2 * (preds * y).sum()) / ((preds + y).sum() + 1e-8)

    print(f"Validation Accuracy: {(num_correct/num_pixels)*100:.2f}%")
    print(f"Validation Dice score: {dice_score/len(loader):.4f}")

    model.train()

def save_preds_as_images(loader, model, folder='saved_images', device='cuda'):
    model.eval()

    if not os.path.exists(folder):
        os.makedirs(folder)

    for idx, (x, y) in enumerate(loader):
        x = x.to(device)
        # Reinstated .unsqueeze(1)
        y = y.to(device).unsqueeze(1)

        with torch.no_grad():
            preds = torch.sigmoid(model(x))
            preds = (preds > 0.5).float()

        torchvision.utils.save_image(preds, f"{folder}/pred_{idx}.png")
        torchvision.utils.save_image(y, f"{folder}/truth_{idx}.png")

        if idx == 2:
            break

    model.train()

In [ ]:
# train1.py:

import torch
import cv2
import os
import glob
import albumentations as A
import torch.optim as optim
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm

# --- DYNAMIC KAGGLE PATH RESOLUTION ---
# Automatically hunt for the Split_Dataset directory to bypass hardcoded slug errors
print("Hunting for dataset directories...")
split_dir_search = glob.glob('/kaggle/input/**/Split_Dataset', recursive=True)

if not split_dir_search:
    raise FileNotFoundError("Could not find 'Split_Dataset' anywhere in /kaggle/input/. Check if the dataset attached correctly.")

OUTPUT_SPLIT_DIR = split_dir_search[0]
print(f"Found Split_Dataset at: {OUTPUT_SPLIT_DIR}")

train_img_dir = os.path.join(OUTPUT_SPLIT_DIR, 'train', 'images')
train_mask_dir = os.path.join(OUTPUT_SPLIT_DIR, 'train', 'masks')
val_img_dir = os.path.join(OUTPUT_SPLIT_DIR, 'val', 'images')
val_mask_dir = os.path.join(OUTPUT_SPLIT_DIR, 'val', 'masks')

WORKING_DIR = "/kaggle/working"

# --- Hyperparameters ---
learning_rate = 5e-5
num_epochs = 100
batch_size = 8
num_workers = 2
pin_memory = True
image_height = 256
image_width = 256
load_model = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_loop(loader, model, optimizer, loss_fn, scaler):
    loop = tqdm(loader)

    for batch_idx, (data, targets) in enumerate(loop):
        data = data.to(device)
        targets = targets.float().unsqueeze(1).to(device)

        with torch.amp.autocast(device_type='cuda'):
            scores = model(data)
            loss = loss_fn(scores, targets) + dice_loss(scores, targets)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        loop.set_postfix(loss=loss.item())

def dice_loss(pred, target):
    pred = torch.sigmoid(pred)
    smooth = 1e-6
    intersection = (pred * target).sum()
    return 1 - ((2 * intersection + smooth) / (pred.sum() + target.sum() + smooth))

In [ ]:
# train2.py: 

def main():
    # 3-Channel logic for Synthetic Pre-training
    train_transform = A.Compose(
        [
            A.Resize(height=image_height, width=image_width, interpolation=cv2.INTER_LINEAR, mask_interpolation=cv2.INTER_NEAREST),
            A.Rotate(limit=15, p=0.5, interpolation=cv2.INTER_LINEAR, mask_interpolation=cv2.INTER_NEAREST),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.1),
            A.Normalize(
                mean=[0.0, 0.0, 0.0],
                std=[1.0, 1.0, 1.0],
                max_pixel_value=255.0,
            ),
            ToTensorV2(),
        ], is_check_shapes=False
    )

    val_transform = A.Compose(
        [
            A.Resize(height=image_height, width=image_width, interpolation=cv2.INTER_LINEAR, mask_interpolation=cv2.INTER_NEAREST),
            A.Normalize(
                mean=[0.0, 0.0, 0.0],
                std=[1.0, 1.0, 1.0],
                max_pixel_value=255.0,
            ),
            ToTensorV2(),
        ], is_check_shapes=False
    )

    model = DilatedCableNet(in_channels=3, out_channels=1).to(device)
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor([10.0]).to(device))
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    train_loader, val_loader = get_loaders(
        train_img_dir, train_mask_dir, train_transform,
        val_img_dir, val_mask_dir, val_transform,
        batch_size, pin_memory, num_workers
    )

    if load_model:
        # We only try to load from the INPUT directory if load_model is True (e.g. resuming)
        # Note: You'll need to manually ensure model.pth.tar exists in your dataset if resuming.
        try:
            load_checkpoint(torch.load(os.path.join(INPUT_DIR, 'model.pth.tar')), model, optimizer)
        except Exception as e:
            print(f"Skipping checkpoint load: {e}")

    scaler = torch.amp.GradScaler()

    for epoch in range(num_epochs):
        model.train()
        print(f"\nProcessing epoch {epoch+1}/{num_epochs}")
        train_loop(train_loader, model, optimizer, loss_fn, scaler)

        checkpoint = {
            "state_dict": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "epoch": epoch
        }
        
        # Save model safely to the WRITABLE Kaggle directory
        save_checkpoint(state=checkpoint, filename=os.path.join(WORKING_DIR, 'model.pth.tar'))

        check_accuracy(loader=val_loader, model=model, device=device)
        
        save_preds_as_images(loader=val_loader, model=model, folder=os.path.join(WORKING_DIR, "saved_images"), device=device)

if __name__ == "__main__":
    main()

In [ ]:
# finetuning.py:

import os
import cv2
import glob
import numpy as np
import shutil
import torch
import torch.optim as optim
import albumentations as A
from albumentations.pytorch import ToTensorV2
from skimage.morphology import skeletonize

# --- 1. Dynamic Kaggle Paths & Configuration ---
WORKING_DIR = "/kaggle/working"

# Dynamically hunt for the Raw_Data folder
print("Hunting for Raw_Data directory...")
raw_dir_search = glob.glob('/kaggle/input/**/Raw_Data', recursive=True)

if not raw_dir_search:
    raise FileNotFoundError("Could not find 'Raw_Data' anywhere in /kaggle/input/. Check if the dataset attached correctly.")

RAW_BASE_DIR = raw_dir_search[0]
print(f"Found Raw_Data at: {RAW_BASE_DIR}")

RAW_IMG_DIR = os.path.join(RAW_BASE_DIR, 'images')
RAW_MASK_DIR = os.path.join(RAW_BASE_DIR, 'masks')

FINETUNE_DIR = os.path.join(WORKING_DIR, 'FineTune_Dataset')
FT_IMG_DIR = os.path.join(FINETUNE_DIR, 'images')
FT_MASK_DIR = os.path.join(FINETUNE_DIR, 'masks')

os.makedirs(FT_IMG_DIR, exist_ok=True)
os.makedirs(FT_MASK_DIR, exist_ok=True)

# --- 2. Morphological Skeletonization ---
def generate_pixel_perfect_masks():
    print("Skeletonizing thick masks to 1-pixel centerlines...")
    raw_images = [f for f in os.listdir(RAW_IMG_DIR) if f.endswith('.jpg')]
    
    for img_name in raw_images:
        mask_name = f"mask_{img_name.replace('.jpg', '.png')}"
        mask_path = os.path.join(RAW_MASK_DIR, mask_name)
        
        thick_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if thick_mask is None:
            print(f"Warning: Could not read mask {mask_path}")
            continue
            
        # Binarize and skeletonize
        _, binary = cv2.threshold(thick_mask, 127, 1, cv2.THRESH_BINARY)
        skeleton = skeletonize(binary)
        perfect_mask = (skeleton * 255).astype(np.uint8)

        # 3-pixel spatial tolerance
        kernel = np.ones((3, 3), np.uint8)
        perfect_mask = cv2.dilate(perfect_mask, kernel, iterations=1)
        
        # Save to writable working directory
        shutil.copy2(os.path.join(RAW_IMG_DIR, img_name), os.path.join(FT_IMG_DIR, img_name))
        
        new_mask_name = img_name.replace('.jpg', '.png')
        cv2.imwrite(os.path.join(FT_MASK_DIR, new_mask_name), perfect_mask)
        
    print(f"Successfully generated {len(raw_images)} 1-pixel ground truth masks.")

# --- 3. Fine-Tuning Execution ---
def run_finetuning():
    # CRITICAL FIX 1: Dropped to 5e-6 for smooth convergence
    FT_LEARNING_RATE = 5e-6 
    FT_EPOCHS = 30
    BATCH_SIZE = 4 

    train_transform = A.Compose([
        A.Resize(height=256, width=256, interpolation=cv2.INTER_LINEAR, mask_interpolation=cv2.INTER_NEAREST),
        A.Rotate(limit=15, p=0.5, interpolation=cv2.INTER_LINEAR, mask_interpolation=cv2.INTER_NEAREST),
        A.HorizontalFlip(p=0.5),
        A.Normalize(mean=[0.0, 0.0, 0.0], std=[1.0, 1.0, 1.0], max_pixel_value=255.0),
        ToTensorV2(),
    ], is_check_shapes=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = DilatedCableNet(in_channels=3, out_channels=1).to(device)
    optimizer = optim.Adam(model.parameters(), lr=FT_LEARNING_RATE)
    
    # Weight is correctly set to 25.0
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor([25.0]).to(device))
    scaler = torch.amp.GradScaler()

    # CRITICAL FIX 2: Strict Weight Reset Logic
    # Forces the script to completely ignore /kaggle/working/ and only pull from your uploaded dataset
    print("Hunting for PRISTINE pre-trained weights strictly in /kaggle/input/...")
    model_search = glob.glob('/kaggle/input/**/**/model.pth.tar', recursive=True)

    if model_search:
        checkpoint_path = model_search[0]
        print(f"Found pristine weights at: {checkpoint_path}")
        load_checkpoint(torch.load(checkpoint_path), model, optimizer)
    else:
        raise FileNotFoundError("CRITICAL: Could not find your pristine model.pth.tar in the input dataset! Did you attach it?")

    # Train and validate on the same 36 images to force domain overfitting
    train_loader, val_loader = get_loaders(
        FT_IMG_DIR, FT_MASK_DIR, train_transform, 
        FT_IMG_DIR, FT_MASK_DIR, train_transform, 
        BATCH_SIZE, True, 2
    )

    for epoch in range(FT_EPOCHS):
        model.train()
        print(f"\nFine-Tuning Epoch {epoch+1}/{FT_EPOCHS}")
        train_loop(train_loader, model, optimizer, loss_fn, scaler)

        checkpoint = {
            "state_dict": model.state_dict(),
            "optimizer": optimizer.state_dict(), 
            "epoch": epoch
        }
        # Save as a distinct fine-tuned model
        save_checkpoint(state=checkpoint, filename=os.path.join(WORKING_DIR, 'finetuned_model.pth.tar'))

        check_accuracy(loader=val_loader, model=model, device=device)

# Execute the pipeline
generate_pixel_perfect_masks()
run_finetuning()

In [ ]:
import matplotlib.pyplot as plt
import cv2
import os

# Paths
SAVED_IMAGES_DIR = '/kaggle/working/saved_images'

def visualize_latest_predictions():
    if not os.path.exists(SAVED_IMAGES_DIR):
        print("No saved images found. Did the training loop complete?")
        return

    # Find all prediction images (they start with 'pred_')
    pred_files = [f for f in os.listdir(SAVED_IMAGES_DIR) if f.startswith('pred_')]
    
    if not pred_files:
        print("No prediction files found in the directory.")
        return

    # Sort them to get the latest ones (assuming you want to see a few samples)
    pred_files.sort()
    num_to_plot = min(3, len(pred_files))
    
    fig, axes = plt.subplots(num_to_plot, 2, figsize=(10, 4 * num_to_plot))
    
    # Handle single image edge case
    if num_to_plot == 1:
        axes = np.expand_dims(axes, axis=0)

    for i in range(num_to_plot):
        pred_filename = pred_files[i]
        # The corresponding truth file has the same index
        truth_filename = pred_filename.replace('pred_', 'truth_')
        
        pred_path = os.path.join(SAVED_IMAGES_DIR, pred_filename)
        truth_path = os.path.join(SAVED_IMAGES_DIR, truth_filename)
        
        pred_img = cv2.imread(pred_path, cv2.IMREAD_GRAYSCALE)
        truth_img = cv2.imread(truth_path, cv2.IMREAD_GRAYSCALE)
        
        axes[i, 0].imshow(truth_img, cmap='gray')
        axes[i, 0].set_title(f"Ground Truth (Skeleton): {truth_filename}")
        axes[i, 0].axis('off')
        
        axes[i, 1].imshow(pred_img, cmap='gray')
        axes[i, 1].set_title(f"Network Prediction: {pred_filename}")
        axes[i, 1].axis('off')

    plt.tight_layout()
    plt.show()

visualize_latest_predictions()

In [ ]:
!pip install -q git+https://github.com/facebookresearch/segment-anything-2.git

In [ ]:
import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from skimage.morphology import skeletonize
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

def evaluate_sam2_and_calculate_sag(img_path, gt_mask_path, sam2_checkpoint):
    print(f"Loading Image: {os.path.basename(img_path)}")
    img_bgr = cv2.imread(img_path)
    gt_mask = cv2.imread(gt_mask_path, cv2.IMREAD_GRAYSCALE)
    
    if img_bgr is None or gt_mask is None:
        print("Error: Could not load image or Ground Truth mask.")
        return
        
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    
   # --- 1. AUTONOMOUS PROMPT GENERATION (Spatial Sorting + Buffered Halo) ---
    gt_binary_img = (gt_mask > 0).astype(np.uint8) * 255
    
    # Extract Positive Points
    y_cable, x_cable = np.where(gt_binary_img == 255)
    if len(x_cable) == 0:
        print("Ground truth mask is empty.")
        return
        
    cable_coords = np.column_stack((x_cable, y_cable))
    
    # FIX 1: Sort by X-coordinate to guarantee true left-to-right spatial distribution
    cable_coords = cable_coords[np.argsort(cable_coords[:, 0])]
    
    # Pick 10 evenly spaced positive points from left to right
    step = max(1, len(cable_coords) // 10)
    pos_points = cable_coords[::step][:10]
    pos_labels = np.ones(len(pos_points), dtype=np.int32) # Label 1 = Foreground

    # FIX 2: The Buffered Halo (Prevent negative points from touching anti-aliased edges)
    kernel_small = np.ones((5, 5), np.uint8)   # The 5-pixel Buffer Zone (Do Not Touch)
    kernel_large = np.ones((25, 25), np.uint8) # The 25-pixel Outer Boundary
    
    dilated_small = cv2.dilate(gt_binary_img, kernel_small, iterations=1)
    dilated_large = cv2.dilate(gt_binary_img, kernel_large, iterations=1)
    
    # Halo is strictly the area OUTSIDE the buffer zone
    halo_mask = cv2.subtract(dilated_large, dilated_small)
    
    y_halo, x_halo = np.where(halo_mask == 255)
    halo_coords = np.column_stack((x_halo, y_halo))
    
    # Pick 20 negative points in the safe background halo
    np.random.shuffle(halo_coords)
    neg_points = halo_coords[:20]
    neg_labels = np.zeros(len(neg_points), dtype=np.int32) # Label 0 = Background

    # Combine into SAM 2 format
    input_points = np.vstack((pos_points, neg_points))
    input_labels = np.concatenate((pos_labels, neg_labels))
    # --- 2. SAM 2 INFERENCE (Point Prompting) ---
    print("Executing Multi-Point SAM 2 Inference...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    sam2_model = build_sam2("sam2_hiera_l.yaml", sam2_checkpoint, device=device)
    predictor = SAM2ImagePredictor(sam2_model)
    
    predictor.set_image(img_rgb)
    # Feed the points instead of the box
    masks, _, _ = predictor.predict(
        point_coords=input_points,
        point_labels=input_labels,
        multimask_output=False
    )
    sam_mask_binary = (masks[0] * 1).astype(np.uint8)

   # --- 3. STANDARD EVALUATION: RELAXED IoU CALCULATION ---
    gt_binary = (gt_mask > 0).astype(np.uint8)
    
    # Dilate the 1-pixel GT mask to match the ~3-pixel Anti-Aliased RGB thickness
    kernel_iou = np.ones((3, 3), np.uint8)
    gt_relaxed = cv2.dilate(gt_binary, kernel_iou, iterations=1)
    
    intersection = np.logical_and(sam_mask_binary, gt_relaxed).sum()
    union = np.logical_or(sam_mask_binary, gt_relaxed).sum()
    iou_score = intersection / union if union > 0 else 0.0
    print(f"Evaluation Metric -> Relaxed SAM 2 IoU Score: {iou_score * 100:.2f}%")

    # --- 4. KINEMATIC STATE (SAG CALCULATION) ---
    print("Extracting Kinematics...")
    skeleton = skeletonize(sam_mask_binary)
    y_coords, x_coords = np.where(skeleton)
    
    try:
        a, b, c = np.polyfit(x_coords, y_coords, 2)
        
        x_start, x_end = np.min(x_coords), np.max(x_coords)
        y_start = a * (x_start**2) + b * x_start + c
        y_end = a * (x_end**2) + b * x_end + c
        
        chord_slope = (y_end - y_start) / (x_end - x_start)
        chord_intercept = y_start - (chord_slope * x_start)
        
        x_sag = (chord_slope - b) / (2 * a)
        y_parabola_at_sag = a * (x_sag**2) + b * x_sag + c
        y_chord_at_sag = chord_slope * x_sag + chord_intercept
        sag_depth_pixels = abs(y_parabola_at_sag - y_chord_at_sag)
        
        print(f"Polynomial: y = {a:.6f}x² + {b:.4f}x + {c:.2f}")
        print(f"Maximum Sag Depth: {sag_depth_pixels:.2f} pixels")

        # --- 5. VISUALIZATION ---
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        
        # Plot 1: The Segmentation Evaluation
        axes[0].imshow(img_rgb)
        overlay_gt = np.zeros_like(img_rgb)
        overlay_gt[gt_binary == 1] = [0, 255, 0] # Ground Truth is Green
        overlay_sam = np.zeros_like(img_rgb)
        overlay_sam[sam_mask_binary == 1] = [255, 0, 0] # SAM 2 is Red
        
        axes[0].imshow(overlay_gt, alpha=0.5)
        axes[0].imshow(overlay_sam, alpha=0.5)
        axes[0].set_title(f"SAM 2 Prediction (Red) vs Ground Truth (Green) | IoU: {iou_score * 100:.2f}%")
        axes[0].axis('off')

        # Plot 2: The Kinematic Measurement
        axes[1].imshow(img_rgb)
        x_line = np.linspace(x_start, x_end, 100)
        y_line = a * (x_line**2) + b * x_line + c
        axes[1].plot(x_line, y_line, color='cyan', linewidth=2, label='Polynomial Fit')
        axes[1].plot([x_start, x_end], [y_start, y_end], color='red', linestyle='--', label="Chord")
        axes[1].plot([x_sag, x_sag], [y_chord_at_sag, y_parabola_at_sag], color='yellow', linewidth=3, label=f"Max Sag: {sag_depth_pixels:.1f} px")
        axes[1].set_title(f"Mathematical Sag Measurement: {sag_depth_pixels:.1f} px")
        axes[1].legend()
        axes[1].axis('off')

        plt.tight_layout()
        plt.show()

    except np.linalg.LinAlgError:
        print("Math failed to converge.")

# --- EXECUTE ---
# Update these paths to point to a specific synthetic image and its matching mask in Kaggle
IMG_PATH = "/kaggle/input/datasets/tejasjoshi2005/cable-robot-dataset/Robotics_Project/Split_Dataset/train/images/image_0_bg0_syn0.jpg"
MASK_PATH = "/kaggle/input/datasets/tejasjoshi2005/cable-robot-dataset/Robotics_Project/Split_Dataset/train/masks/image_0_bg0_syn0.png"
SAM2_PT = "/kaggle/input/models/metaresearch/segment-anything-2/pytorch/sam2-hiera-large/1/sam2_hiera_large.pt"

evaluate_sam2_and_calculate_sag(IMG_PATH, MASK_PATH, SAM2_PT)

In [ ]:
import os
import cv2
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from skimage.morphology import skeletonize
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

def process_single_image(img_path, gt_mask_path, predictor):
    """Silently processes an image and returns the metrics."""
    img_bgr = cv2.imread(img_path)
    gt_mask = cv2.imread(gt_mask_path, cv2.IMREAD_GRAYSCALE)
    
    if img_bgr is None or gt_mask is None:
        return None
        
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    
    # --- 1. PROMPT GENERATION ---
    gt_binary_img = (gt_mask > 0).astype(np.uint8) * 255
    y_cable, x_cable = np.where(gt_binary_img == 255)
    
    if len(x_cable) == 0:
        return None
        
    cable_coords = np.column_stack((x_cable, y_cable))
    cable_coords = cable_coords[np.argsort(cable_coords[:, 0])]
    
    step = max(1, len(cable_coords) // 10)
    pos_points = cable_coords[::step][:10]
    pos_labels = np.ones(len(pos_points), dtype=np.int32)
    
    kernel_small = np.ones((5, 5), np.uint8)
    kernel_large = np.ones((25, 25), np.uint8)
    dilated_small = cv2.dilate(gt_binary_img, kernel_small, iterations=1)
    dilated_large = cv2.dilate(gt_binary_img, kernel_large, iterations=1)
    halo_mask = cv2.subtract(dilated_large, dilated_small)
    
    y_halo, x_halo = np.where(halo_mask == 255)
    halo_coords = np.column_stack((x_halo, y_halo))
    np.random.shuffle(halo_coords)
    neg_points = halo_coords[:20]
    neg_labels = np.zeros(len(neg_points), dtype=np.int32)
    
    input_points = np.vstack((pos_points, neg_points))
    input_labels = np.concatenate((pos_labels, neg_labels))

    # --- 2. SAM 2 INFERENCE ---
    predictor.set_image(img_rgb)
    masks, _, _ = predictor.predict(
        point_coords=input_points,
        point_labels=input_labels,
        multimask_output=False
    )
    sam_mask_binary = (masks[0] * 1).astype(np.uint8)

    # --- 3. RELAXED IoU ---
    gt_binary = (gt_mask > 0).astype(np.uint8)
    kernel_iou = np.ones((3, 3), np.uint8)
    gt_relaxed = cv2.dilate(gt_binary, kernel_iou, iterations=1)
    
    intersection = np.logical_and(sam_mask_binary, gt_relaxed).sum()
    union = np.logical_or(sam_mask_binary, gt_relaxed).sum()
    iou_score = intersection / union if union > 0 else 0.0

    # --- 4. KINEMATICS ---
    skeleton = skeletonize(sam_mask_binary)
    y_coords, x_coords = np.where(skeleton)
    
    try:
        a, b, c = np.polyfit(x_coords, y_coords, 2)
        x_start, x_end = np.min(x_coords), np.max(x_coords)
        y_start = a * (x_start**2) + b * x_start + c
        y_end = a * (x_end**2) + b * x_end + c
        
        chord_slope = (y_end - y_start) / (x_end - x_start)
        chord_intercept = y_start - (chord_slope * x_start)
        
        x_sag = (chord_slope - b) / (2 * a)
        y_parabola_at_sag = a * (x_sag**2) + b * x_sag + c
        y_chord_at_sag = chord_slope * x_sag + chord_intercept
        sag_depth_pixels = abs(y_parabola_at_sag - y_chord_at_sag)
        
        return {
            "IoU": round(iou_score * 100, 2),
            "Sag_Pixels": round(sag_depth_pixels, 2),
            "Poly_a": a,
            "Poly_b": b,
            "Poly_c": c
        }
    except np.linalg.LinAlgError:
        return None

# --- BATCH EXECUTION ---
def run_evaluation_pipeline(images_dir, masks_dir, sam2_checkpoint):
    print("Booting SAM 2 Model for Batch Inference...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    sam2_model = build_sam2("sam2_hiera_l.yaml", sam2_checkpoint, device=device)
    predictor = SAM2ImagePredictor(sam2_model)
    
    results = []
    
    # Grab all JPG files in your images directory
    image_files = [f for f in os.listdir(images_dir) if f.endswith('.jpg')]
    print(f"Found {len(image_files)} images. Starting pipeline...")
    
    # Wrap in tqdm for a progress bar
    for img_name in tqdm(image_files):
        img_path = os.path.join(images_dir, img_name)
        
        # Match the image name to your mask naming convention
        # Change this string manipulation if your mask names are formatted differently!
        mask_name = img_name.replace('.jpg', '.png')
        gt_mask_path = os.path.join(masks_dir, mask_name)
        
        # Ensure the mask actually exists before processing
        if not os.path.exists(gt_mask_path):
            mask_name = f"mask_{img_name.replace('.jpg', '.png')}"
            gt_mask_path = os.path.join(masks_dir, mask_name)
            
        if os.path.exists(gt_mask_path):
            metrics = process_single_image(img_path, gt_mask_path, predictor)
            
            if metrics:
                results.append({
                    "Filename": img_name,
                    "IoU_Score": metrics["IoU"],
                    "Calculated_Sag": metrics["Sag_Pixels"],
                    "Coefficient_A": metrics["Poly_a"],
                    "Coefficient_B": metrics["Poly_b"],
                    "Coefficient_C": metrics["Poly_c"]
                })
        else:
            print(f"Skipping {img_name} - Could not find matching mask.")

    # Export to a Pandas DataFrame and save to CSV
    df = pd.DataFrame(results)
    
    # Calculate System-Wide Averages
    mean_iou = df['IoU_Score'].mean()
    print(f"\nPipeline Complete!")
    print(f"Total Successfully Processed: {len(df)}")
    print(f"Average System IoU: {mean_iou:.2f}%")
    
    csv_path = "SAM2_Evaluation_Results.csv"
    df.to_csv(csv_path, index=False)
    print(f"All data successfully exported to: {csv_path}")

# --- EXECUTE ---
IMAGES_DIR = "/kaggle/input/datasets/tejasjoshi2005/cable-robot-dataset/Robotics_Project/Split_Dataset/train/images"
MASKS_DIR = "/kaggle/input/datasets/tejasjoshi2005/cable-robot-dataset/Robotics_Project/Split_Dataset/train/masks"
SAM2_PT = "/kaggle/input/models/metaresearch/segment-anything-2/pytorch/sam2-hiera-large/1/sam2_hiera_large.pt"

run_evaluation_pipeline(IMAGES_DIR, MASKS_DIR, SAM2_PT)